In [ ]:
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import h5py

class FileManager:
    def __init__(self, project_name="Nro_Item"):
        # 1. Definir Rutas
        self.base = Path.cwd() / project_name
        self.raw = self.base / "01 Respaldo"
        self.analysis = self.base / "02 Analisis de datos"
        self.results = self.base / "03 Resultados"
        
        # 2. Crear Estructura Automáticamente al iniciar
        self._create_structure()
    
    def _create_structure(self):
        """Crea las carpetas si no existen"""
        for carpeta in [self.raw, self.analysis, self.results]:
            carpeta.mkdir(parents=True, exist_ok=True)
            print(f"Verificado: {carpeta}")

    def get_new_name(self, prefijo="medicion", extension=".bin"):
        """Genera una ruta con timestamp para no sobrescribir nunca"""
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        nombre = f"{prefijo}_{timestamp}{extension}"
        
        if extension == ".bin":
            return self.raw / nombre
        elif extension == ".h5":
            return self.analysis / nombre
        elif extension == ".png" or extension == ".pdf":
            return self.results / nombre
        else:
            return self.base / nombre

# Señales de ejemplo 
def generar_senales():
    dt = 20e-9
    num_points = 4000
    time_axis = np.linspace(0, (num_points - 1) * dt, num_points)
    ch1 = np.sin(2 * np.pi * 1e5 * time_axis)
    # Convertir a binario int16 con escala 25 muestras/V
    ch1_bin = 25 * ch1
    ch1_bin = ch1_bin.astype('>i2').tobytes()
    return ch1, ch1_bin, time_axis

# Crear BIN (int16): para guardar datos originales como copia de seguridad.
def create_bin_int16(data, file_path):
    with open(file_path, "wb") as f:
        f.write(data)
        print(f"Se creó '{file_path}'")

# Crear HDF5: para almacenamiento principal y procesamiento.
def create_hdf5(time, ch1, file_path):
    # Guarda donde tú le digas
    with h5py.File(file_path, "w") as f:
        f.create_dataset("Tiempo [s]", data=time, compression="gzip")
        f.create_dataset("Tensión [V]", data=ch1, compression="gzip")
    print(f"Se creó '{file_path}'")

# Crear CSV: para exportar datos.
def create_csv(time, tension, file_path):
    # Crear el DataFrame con Nombres de Columnas.
    df = pd.DataFrame({
        "Tiempo [s]": time,
        "Tensión [V]": tension
    })
    # float_format='%.4E' guarda en notación científica (ej: 1.2345E-03).
    df.to_csv(file_path, index=False, sep=',', float_format='%.4E')
    print(f"Se creó '{file_path}'")

if __name__ == '__main__':
    # 1. Al iniciar tu software, inicializas el gestor
    gestor = FileManager()
    ch1, ch1_bin, time_axis = generar_senales()
    
    # 2. Guardar datos originales en BIN:
    ruta_bin = gestor.get_new_name("Onda", ".bin")
    create_bin_int16(ch1_bin, ruta_bin)
    print(f"Se guardó datos originales en: {ruta_bin}")
    
    # 3. Guardar datos procesados:
    # Usamos el mismo nombre base (stem) del binario para mantener trazabilidad
    nombre_base = ruta_bin.stem 
    ruta_h5 = gestor.analysis / f"{nombre_base}_procesado.h5"
    create_hdf5(time_axis, ch1, ruta_h5)
    print(f"Se guardó datos procesados en: {ruta_h5}")

    # 4. Exportar datos a CSV:
    ruta_csv = gestor.results / f"{nombre_base}_exportado.csv"
    create_csv(time_axis, ch1, ruta_csv)
    print(f"Se guardó datos exportados en: {ruta_csv}")